# Agentic RAG Finance Demo

**Business Problem:**  
We need an AI-driven Retrieval-Augmented Generation (RAG) system to answer finance queries with source citations, leveraging large financial documents.

**Data Files Used available in folder ./financial_docs:**  
- `Q1_2024_Financial_Report.txt`  
- `Market_Analysis_2024.txt`  

**Program Flow:**  
1. Load, split, embed, and index documents in Chroma.  
2. Define RAG and summarization tools using Cohere LLM.  
3. Initialize a Zero-Shot agent with multiple tools.  
4. Provide a CLI interface for interactive querying.  

**Agentic RAG Theory:**  
The agent uses LangChain’s RetrievalQA to fetch relevant document chunks via embeddings, then a Cohere LLM to generate answers with citations, wrapped in a tool. A Zero-Shot React Description agent selects the correct tool per query.

## Import libraries

In [ ]:

import os
import shutil
import argparse

from langchain_cohere import ChatCohere
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings
from langchain_chroma import Chroma
from langchain.chains import RetrievalQA
from langchain.agents import Tool, initialize_agent, AgentType

## Load, Split, Embed, and Index Documents

In [ ]:
# Load, Split, Embed, and Index Documents

import oci
from LoadProperties import LoadProperties
properties=LoadProperties()

from langchain_community.embeddings import OCIGenAIEmbeddings
embeddings = OCIGenAIEmbeddings(
    model_id=properties.getEmbeddingModelName(),
    service_endpoint=properties.getEndpoint(),
    compartment_id=properties.getCompartment(),
    auth_type='INSTANCE_PRINCIPAL',
)

# Read data files
docs_folder = "./financial_docs"  

loaders = [TextLoader(os.path.join(docs_folder, f), encoding="utf8")
           for f in os.listdir(docs_folder) if f.endswith(".txt")]

all_docs = []
for loader in loaders:
    all_docs.extend(loader.load())

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(all_docs)

vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chromadb")

## Build RetrievalQA and Define RAG Tool

In [ ]:
# Build RetrievalQA and Define RAG Tool
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI

llm = ChatOCIGenAI(
      model_id='meta.llama-3.3-70b-instruct',
      service_endpoint=properties.getEndpoint(),
      compartment_id=properties.getCompartment(),auth_type='INSTANCE_PRINCIPAL',
      model_kwargs={ "max_tokens": 600},)



qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k":3}),
    return_source_documents=True
)

def financial_qa(query: str) -> str:
    """Tool function: runs QA and appends source citations."""
    result = qa_chain({"query": query})
    answer = result["result"]
    sources = result["source_documents"]
    citation_lines = []
    for doc in sources:
        src = os.path.basename(doc.metadata.get("source", "unknown"))
        citation_lines.append(f"- {src} (page chunk)")
    citations = "\n".join(citation_lines)
    return f"{answer}\n\nSources:\n{citations}"

## Register Tool and Initialize Agent

In [ ]:
# Define Summarization Tools

tools = [
    Tool(
        name="FinancialRAG",
        func=financial_qa,
        description="Use for answering questions about our Q1 2024 financials and market analysis, with source references."
    )
]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, handle_parsing_errors=True,
    verbose=True
)

## Pass a query to the Agent 
### Query1

In [ ]:
query = "Summarize our net income performance in Q1 2024."

response = agent.invoke(query)
print("\n=== Agent Response ===\n")

from IPython.display import Markdown, display
print(response['input'],"\n")
print(response['output'])

### Query2

In [ ]:
query = "What was our YoY revenue growth in Q1 2024?"

response = agent.invoke(query)
print("\n=== Agent Response ===\n")

from IPython.display import Markdown, display
print(response['input'],"\n")
print(response['output'])

### Query3

In [ ]:
query = "According to the Market_Analysis_2024 report, what were the key market trends observed in Q1 2024?"

response = agent.invoke(query)
print("\n=== Agent Response ===\n")

from IPython.display import Markdown, display
print(response['input'],"\n")
display((Markdown(response['output'])))

### Query4

In [ ]:
# Section 8
query = "What was our YoY revenue growth in Q1 2024, and what market volatility trends were noted in the Market_Analysis_2024 report?"

response = agent.invoke(query)
print("\n=== Agent Response ===\n")

from IPython.display import Markdown, display
print(response['input'],"\n")
display((Markdown(response['output'])))